# Next-POI recommendation — TLMR-faithful baseline

Foursquare NYC, end-to-end on Google Colab T4. Mirrors `implementation_guide.md` sections 2–9. The notebook is a thin wrapper: every function lives in the `src/` package; cells here just orchestrate.

**Expected runtime:** ~1.5–2 hours on free T4 (NYC only). **Expected results:** HR@1 ≈ 0.13–0.18 (LSTM/STGCN tier from LLM4POI Table 3).

Before running, set Runtime → Change runtime type → **T4 GPU**.

> **Note:** TKY is intentionally out of scope for this run. To add it later, re-introduce a `df_tky` load + `preprocess_dataset(df_tky, 'TKY')` + `train_model('TKY', ...)` cell.

## 2. Colab setup

### 2.1 Install packages
PyG 2.5+ installs cleanly without separate `torch_scatter` / `torch_sparse` wheels.

In [ ]:
!pip install -q torch_geometric==2.5.3 tensorboardX==2.6.2.2 pyarrow==15.0.2
print('Done. Runtime should NOT need to restart.')

### 2.2 Mount Drive and set up paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/poi-rec'
os.makedirs(PROJECT_ROOT, exist_ok=True)
for sub in ['data/raw', 'data/processed', 'checkpoints', 'results']:
    os.makedirs(os.path.join(PROJECT_ROOT, sub), exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(os.listdir(PROJECT_ROOT))

### 2.3 Make `src/` importable

Choose **one** option below. Recommended: clone from GitHub. Alternatives: upload `src/` to Drive, or zip-upload through the Files panel.

**Option A — GitHub clone (recommended).** Replace the URL with this repo's URL.

In [ ]:
# Option A: clone from GitHub. Replace REPO_URL with your fork.
REPO_URL = 'https://github.com/YOUR-USERNAME/PFE_IMPLEMTATION.git'
REPO_DIR = '/content/PFE_IMPLEMTATION'

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Sanity: import a public symbol
from src.models.next_poi import NextPOIModel
print('src/ on path:', REPO_DIR)

**Option B — Drive copy.** If you uploaded the whole project to Drive at `MyDrive/poi-rec-code/`, run this cell instead of Option A. Skip otherwise.

In [ ]:
# Option B: load src/ from Drive. Comment out if using Option A.
# import sys
# DRIVE_CODE = '/content/drive/MyDrive/poi-rec-code'
# if DRIVE_CODE not in sys.path:
#     sys.path.insert(0, DRIVE_CODE)
# from src.models.next_poi import NextPOIModel
# print('src/ on path:', DRIVE_CODE)

### 2.4 Imports + reproducibility

Single seed (42) for python, numpy, torch, and CUDA — matches `implementation_guide.md`.

In [ ]:
import os, json, random
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 3. Dataset download

### 3.1 Download Yang et al. (2015) Foursquare NYC

The zip ships both NYC and TKY .txt files; we only load NYC below.

In [ ]:
import urllib.request, zipfile

URL = 'http://www-public.imtbs-tsp.eu/~zhang_da/pub/dataset_tsmc2014.zip'
RAW_DIR = os.path.join(PROJECT_ROOT, 'data/raw')
ZIP_PATH = os.path.join(RAW_DIR, 'dataset_tsmc2014.zip')

if not os.path.exists(ZIP_PATH):
    print('Downloading... ~30MB')
    urllib.request.urlretrieve(URL, ZIP_PATH)
    print('Done.')

if not os.path.exists(os.path.join(RAW_DIR, 'dataset_TSMC2014_NYC.txt')):
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(RAW_DIR)
    print('Extracted.')

print(os.listdir(RAW_DIR))

**If the URL 404s:** download from the GETNext repo (`https://github.com/songyangco/GETNext`, look in `data/`) or Kaggle, then upload to `data/raw/` via the Colab Files panel.

### 3.2 Load and inspect

In [ ]:
from src.data.preprocess import load_foursquare

df_nyc = load_foursquare(os.path.join(RAW_DIR, 'dataset_TSMC2014_NYC.txt'))

print(f"NYC: {len(df_nyc):,} check-ins | "
      f"{df_nyc['user_id'].nunique():,} users | "
      f"{df_nyc['poi_id'].nunique():,} POIs | "
      f"{df_nyc['timestamp'].min().date()} \u2192 {df_nyc['timestamp'].max().date()}")

## 4. Preprocessing

Filter (≥10 check-ins per user, ≥10 visitors per POI, iterative until stable) → re-index to `0..N-1` → sessionize at 24h gaps with Δd, Δt → drop singletons → 70/10/20 chronological split per user. All implemented in `src.data.preprocess`.

In [ ]:
from src.data.preprocess import (
    iterative_filter, reindex, build_sessions, chronological_split,
)

def preprocess_dataset(df_raw, name):
    print(f'\n=== {name} ===')
    print(f"Raw: {len(df_raw):,} check-ins, "
          f"{df_raw['user_id'].nunique()} users, "
          f"{df_raw['poi_id'].nunique()} POIs")

    df = iterative_filter(df_raw)
    print(f"Filtered: {len(df):,} check-ins, "
          f"{df['user_id'].nunique()} users, "
          f"{df['poi_id'].nunique()} POIs")

    df, user2idx, poi2idx = reindex(df)
    df = build_sessions(df)
    print(f"Sessionized: {df['session_id'].nunique():,} sessions, "
          f"avg length {df.groupby('session_id').size().mean():.1f}")

    train, val, test = chronological_split(df)
    print(f"Train: {train['session_id'].nunique()} sessions ({len(train)} check-ins)")
    print(f"Val:   {val['session_id'].nunique()} sessions ({len(val)} check-ins)")
    print(f"Test:  {test['session_id'].nunique()} sessions ({len(test)} check-ins)")

    out = os.path.join(PROJECT_ROOT, 'data/processed', name)
    os.makedirs(out, exist_ok=True)
    train.to_parquet(os.path.join(out, 'train.parquet'))
    val.to_parquet(os.path.join(out, 'val.parquet'))
    test.to_parquet(os.path.join(out, 'test.parquet'))

    poi_coords = (df.groupby('poi_idx')[['lat', 'lon']].first()
                    .sort_index().values)
    np.save(os.path.join(out, 'poi_coords.npy'), poi_coords)

    meta = {
        'n_users': len(user2idx),
        'n_pois': len(poi2idx),
        'n_train_sessions': int(train['session_id'].nunique()),
        'n_val_sessions':   int(val['session_id'].nunique()),
        'n_test_sessions':  int(test['session_id'].nunique()),
    }
    with open(os.path.join(out, 'meta.json'), 'w') as f:
        json.dump(meta, f, indent=2)

    return train, val, test, meta

train_nyc, val_nyc, test_nyc, meta_nyc = preprocess_dataset(df_nyc, 'NYC')

## 5. Hybrid graph construction

Co-visit edges (≥3 transitions in **training data only**, no leakage) ∪ kNN edges (k=10, haversine via `BallTree`). Stored as a symmetric PyG `edge_index`.

In [ ]:
from src.data.graph import build_hybrid_graph

def make_and_save_graph(name, train_df, meta):
    print(f'\n=== Building graph for {name} ===')
    out = os.path.join(PROJECT_ROOT, 'data/processed', name)
    poi_coords = np.load(os.path.join(out, 'poi_coords.npy'))
    edge_index = build_hybrid_graph(train_df, poi_coords, meta['n_pois'])
    torch.save(edge_index, os.path.join(out, 'edge_index.pt'))
    return edge_index

edge_index_nyc = make_and_save_graph('NYC', train_nyc, meta_nyc)

## 6. Model + smoke test

Architecture: 2-layer GCN over POI graph → per-step concat with context (Δd, Δt MLPs) → GRU → concat with user embedding → MLP head → |V| logits.

Spec dimensions (locked): `d_p=128, d_u=64, d_c=32, d_h=128, d_hidden=256, dropout=0.2`.

In [ ]:
import torch.nn.functional as F
from src.models.next_poi import NextPOIModel

def smoke_test():
    n_pois, n_users = 100, 20
    model = NextPOIModel(n_pois, n_users).to(DEVICE)
    edge_index = torch.tensor([[0, 1, 2, 3], [1, 0, 3, 2]],
                              dtype=torch.long, device=DEVICE)
    B, T = 8, 5
    poi_ids = torch.randint(0, n_pois, (B, T), device=DEVICE)
    delta_d = torch.rand(B, T, device=DEVICE)
    delta_t = torch.rand(B, T, device=DEVICE)
    user_ids = torch.randint(0, n_users, (B,), device=DEVICE)
    lengths = torch.tensor([5, 4, 3, 5, 2, 4, 5, 3])

    logits = model(poi_ids, delta_d, delta_t, user_ids, lengths, edge_index)
    print(f'Output shape: {logits.shape}  (expected ({B}, {n_pois}))')
    print(f'After softmax, row sum: {F.softmax(logits, dim=-1).sum(dim=-1)}')
    n_params = sum(p.numel() for p in model.parameters())
    print(f'Total parameters: {n_params:,}')

smoke_test()

## 7. Dataset and DataLoader

Each session of length L gives L-1 (history, target) examples. Histories are padded per batch in the collate function. All implemented in `src.data.dataset`.

In [ ]:
from src.train import make_loaders

# Quick check that the loaders build for NYC.
print('--- NYC loaders ---')
proc = os.path.join(PROJECT_ROOT, 'data/processed', 'NYC')
_ = make_loaders(proc, device=DEVICE, batch_size=64)

## 8. Training

Adam(lr=1e-3, wd=1e-5), batch=64, grad clip 5.0, dropout 0.2, max 50 epochs, early stop on val HR@10 with patience=8. Checkpoints to `checkpoints/NYC/{best,latest}.pt`. Per-epoch metrics to `results/NYC_history.csv`. Implemented in `src.train.train_model`.

### 8.1 NYC (~1.5–2 h on T4)

In [ ]:
from src.train import train_model

model_nyc, test_nyc_metrics, history_nyc = train_model(
    name='NYC', project_root=PROJECT_ROOT, device=DEVICE,
    epochs=50, patience=8,
)

## 9. Reading the results

Compare to LLM4POI Table 3. Target band: HR@1 ∈ [0.13, 0.18] on NYC. Anything > 0.20 with this architecture is suspicious — re-check the chronological split and graph construction for leakage.

In [ ]:
def print_comparison():
    with open(os.path.join(PROJECT_ROOT, 'results', 'NYC_test.json')) as f:
        r = json.load(f)
    print('\n=== NYC ===')
    print(f"  HR@1   = {r['HR@1']:.4f}")
    print(f"  HR@5   = {r['HR@5']:.4f}")
    print(f"  HR@10  = {r['HR@10']:.4f}")
    print(f"  NDCG@5 = {r['NDCG@5']:.4f}")
    print(f"  NDCG@10= {r['NDCG@10']:.4f}")
    print(f"  MRR    = {r['MRR']:.4f}")

print_comparison()
print('\nLiterature reference (HR@1 / Acc@1 on Foursquare NYC, from LLM4POI Table 3):')
print('  LSTM:     NYC=0.13')
print('  STGCN:    NYC=0.18')
print('  STAN:     NYC=0.22')
print('  GETNext:  NYC=0.24')
print('  STHGCN:   NYC=0.27')